# Mat & Comp – ML 10: Retropropagación
## Repartir la culpa hacia atrás

**Curso:** Machine learning y redes neuronales (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 9

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Usar la **regla de la cadena** para calcular cómo cambia una cadena de operaciones.
2. Calcular a mano las pendientes de **una neurona** recorriendo sus cálculos hacia atrás.
3. Programar la **retropropagación** para una red con una capa oculta, con matrices de numpy.
4. **Comprobar** que la retropropagación da las mismas pendientes que el gradiente numérico de ML 9, cientos de veces más rápido.
5. Entrenar una red que reconoce **dígitos escritos a mano** y entender el problema del **desvanecimiento del gradiente**.

## 🔍 Engranajes

En una bicicleta de cambios hay una cadena de engranajes. Supón que:

- cuando el engranaje **A** da una vuelta, el **B** da **3** vueltas;
- cuando el **B** da una vuelta, el **C** da **2** vueltas.

✏️ *Cuando A da una vuelta, ¿cuántas vueltas da C? ¿Y si hubiera un cuarto engranaje D que da 5 vueltas por cada vuelta de C?*

**Mi respuesta:**

Si respondiste multiplicando, ya conoces la **regla de la cadena**: cuando algo depende de algo que depende de algo más, las tasas de cambio **se multiplican**.

En ML 9 entrenamos una red moviendo cada peso un poquito para medir su pendiente. Funcionaba, pero era lentísimo. Hoy vamos a calcular **todas** las pendientes de una sola vez, recorriendo la red **hacia atrás**, como quien sigue la cadena de engranajes desde el final.

## 1. La regla de la cadena

Si $y$ depende de $u$, y $u$ depende de $x$, entonces

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

Ejemplo: $y = (3x + 1)^2$. Llamemos $u = 3x + 1$, así que $y = u^2$.

- $\dfrac{du}{dx} = 3$ (por cada unidad que sube $x$, $u$ sube 3);
- $\dfrac{dy}{du} = 2u$ (la pendiente de $u^2$, que vimos en ML 3);
- entonces $\dfrac{dy}{dx} = 2u \cdot 3 = 6(3x + 1)$.

En $x = 1$: $u = 4$ y la pendiente es $6 \cdot 4 = 24$. Comprobémoslo con la pendiente numérica de ML 3:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return (3 * x + 1) ** 2

x, h = 1.0, 1e-5
numerica = (f(x + h) - f(x - h)) / (2 * h)
cadena = 2 * (3 * x + 1) * 3
print(f"Pendiente numérica: {numerica:.5f}")
print(f"Regla de la cadena: {cadena}")

## 2. La derivada de la sigmoide

Las neuronas de ML 9 usan la sigmoide $\sigma(z) = \dfrac{1}{1 + e^{-z}}$. Su pendiente tiene una forma muy bonita (se demuestra con cálculo, aquí la comprobaremos con números):

$$\sigma'(z) = \sigma(z)\,\big(1 - \sigma(z)\big)$$

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

def d_sigmoide(z):
    s = sigmoide(z)
    return s * (1 - s)

for z in [-4, -1, 0, 1, 4]:
    numerica = (sigmoide(z + h) - sigmoide(z - h)) / (2 * h)
    print(f"z = {z:2}:  fórmula {d_sigmoide(z):.6f}   numérica {numerica:.6f}")

zs = np.linspace(-8, 8, 400)
plt.plot(zs, sigmoide(zs), label="σ(z)")
plt.plot(zs, d_sigmoide(zs), label="σ'(z) = σ(z)(1 − σ(z))")
plt.axhline(0.25, color="gray", linestyle=":")
plt.legend(); plt.grid(alpha=0.3); plt.xlabel("z")
plt.title("La sigmoide y su pendiente")
plt.show()

Observa que la pendiente es **como mucho 0.25** (en z = 0) y casi 0 lejos del centro. Esto será importante en la sección 8.

## 3. Una neurona, hacia adelante y hacia atrás

Tomemos una sola neurona con una entrada. Sus cálculos forman una cadena:

$$x \;\xrightarrow{\;z = w x + b\;}\; z \;\xrightarrow{\;a = \sigma(z)\;}\; a \;\xrightarrow{\;E = (a - y)^2\;}\; E$$

**Hacia adelante** calculamos los valores, de izquierda a derecha. **Hacia atrás** calculamos las pendientes del error, de derecha a izquierda, multiplicando en cada eslabón (como los engranajes):

| Eslabón | Pendiente local |
|---|---|
| $E = (a - y)^2$ | $\dfrac{dE}{da} = 2(a - y)$ |
| $a = \sigma(z)$ | $\dfrac{da}{dz} = \sigma(z)(1 - \sigma(z)) = a(1 - a)$ |
| $z = w x + b$ | $\dfrac{dz}{dw} = x \qquad \dfrac{dz}{db} = 1$ |

Por la regla de la cadena:

$$\frac{dE}{dw} = \underbrace{2(a - y)}_{dE/da} \cdot \underbrace{a(1 - a)}_{da/dz} \cdot \underbrace{x}_{dz/dw} \qquad\qquad \frac{dE}{db} = 2(a - y) \cdot a(1 - a) \cdot 1$$

In [ ]:
x, y = 2.0, 1.0          # un ejemplo: entrada 2, respuesta correcta 1
w, b = -0.5, 0.3         # pesos actuales

# Hacia adelante
z = w * x + b
a = sigmoide(z)
E = (a - y) ** 2
print(f"Adelante:  z = {z:.3f}   a = {a:.4f}   E = {E:.4f}")

# Hacia atrás
dE_da = 2 * (a - y)
da_dz = a * (1 - a)
dE_dz = dE_da * da_dz            # la «culpa» que le toca a z
dE_dw = dE_dz * x
dE_db = dE_dz * 1
print(f"Atrás:     dE/da = {dE_da:.4f}   dE/dz = {dE_dz:.4f}   dE/dw = {dE_dw:.4f}   dE/db = {dE_db:.4f}")

# Comprobación numérica
def E_de(w, b):
    return (sigmoide(w * x + b) - y) ** 2
print(f"Numérico:  dE/dw = {(E_de(w + h, b) - E_de(w - h, b)) / (2 * h):.4f}   dE/db = {(E_de(w, b + h) - E_de(w, b - h)) / (2 * h):.4f}")

¡Coinciden! Y fíjate en algo clave: el valor `dE_dz` se calcula **una vez** y se reutiliza para `w` y para `b`. En una red grande, esa reutilización es lo que ahorra tanto trabajo.

💡 Una forma de pensarlo: `dE_dz` es la **culpa** que le toca a la neurona en el error. Cada peso recibe una parte de esa culpa según la entrada que lo multiplicaba.

## 4. Retropropagación en una red completa

Ahora la red de ML 9: entradas → capa oculta → salida. Usaremos **matrices** para procesar todos los ejemplos a la vez, y dejaremos que la salida tenga varias neuronas (lo usaremos con los dígitos). Por eso las respuestas `Y` serán una tabla: una fila por ejemplo y una columna por neurona de salida.

**Hacia adelante** (igual que en ML 9, guardando los valores intermedios):

$$H = \sigma(X W_1 + b_1) \qquad S = \sigma(H W_2 + b_2) \qquad E = \text{promedio sobre los ejemplos de la suma de } (S - Y)^2$$

(Con una sola salida, como en el XOR, es el mismo error cuadrático medio de ML 9.)

**Hacia atrás**, eslabón por eslabón (con $n$ = número de ejemplos):

| Paso | Fórmula | En numpy |
|---|---|---|
| culpa de la salida | $dS = \dfrac{2}{n}(S - Y)$ | `dS = 2 * (S - Y) / n` |
| atravesar la sigmoide | $dZ_2 = dS \cdot S(1 - S)$ | `dZ2 = dS * S * (1 - S)` |
| pendientes de la capa 2 | $dW_2 = H^T dZ_2$, $\; db_2 = \sum dZ_2$ | `dW2 = H.T @ dZ2` |
| culpa de la capa oculta | $dH = dZ_2 W_2^T$ | `dH = dZ2 @ W2.T` |
| atravesar la sigmoide | $dZ_1 = dH \cdot H(1 - H)$ | `dZ1 = dH * H * (1 - H)` |
| pendientes de la capa 1 | $dW_1 = X^T dZ_1$, $\; db_1 = \sum dZ_1$ | `dW1 = X.T @ dZ1` |

`.T` es la **transpuesta**: cambia filas por columnas. Aparece porque en la ida multiplicamos por $W$ y en la vuelta hay que «deshacer el camino». La regla práctica: **las formas tienen que encajar**, y `dW` debe tener la misma forma que `W`.

Fíjate en el paso «culpa de la capa oculta»: la culpa de la salida se reparte hacia atrás **a través de los mismos pesos** $W_2$. Una neurona oculta con un peso grande hacia la salida recibe más culpa. ¡Eso es la **retro**propagación!

In [ ]:
def iniciar(n_entradas, n_ocultas, n_salidas, semilla):
    rng = np.random.default_rng(semilla)
    return [rng.normal(0, 1, (n_entradas, n_ocultas)), np.zeros(n_ocultas),
            rng.normal(0, 1, (n_ocultas, n_salidas)), np.zeros(n_salidas)]

def adelante(X, params):
    W1, b1, W2, b2 = params
    H = sigmoide(X @ W1 + b1)          # (n, ocultas)
    S = sigmoide(H @ W2 + b2)          # (n, salidas)
    return H, S

def error(X, Y, params):
    _, S = adelante(X, params)
    return np.mean(np.sum((S - Y) ** 2, axis=1))   # suma sobre las salidas, promedio sobre los ejemplos

def retropropagacion(X, Y, params):
    W1, b1, W2, b2 = params
    n = len(X)
    H, S = adelante(X, params)
    dS = 2 * (S - Y) / n
    dZ2 = dS * S * (1 - S)
    dW2 = H.T @ dZ2
    db2 = dZ2.sum(axis=0)
    dH = dZ2 @ W2.T
    dZ1 = dH * H * (1 - H)
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0)
    return [dW1, db1, dW2, db2]

## 5. ¿Está bien? Comprobación con el gradiente numérico

Es muy fácil equivocarse en una fórmula (un signo, una transpuesta…). Los programadores profesionales siempre comprueban la retropropagación contra el gradiente numérico de ML 9 en una red pequeña:

In [ ]:
def gradiente_numerico(X, Y, params, h=1e-5):
    gradientes = []
    for A in params:
        G = np.zeros_like(A)
        for pos in np.ndindex(A.shape):
            original = A[pos]
            A[pos] = original + h; e_mas = error(X, Y, params)
            A[pos] = original - h; e_menos = error(X, Y, params)
            A[pos] = original
            G[pos] = (e_mas - e_menos) / (2 * h)
        gradientes.append(G)
    return gradientes

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
Y = np.array([[0], [1], [1], [0]], dtype=float)        # XOR, como columna

params = iniciar(2, 3, 1, semilla=0)
g_retro = retropropagacion(X, Y, params)
g_num = gradiente_numerico(X, Y, params)

for nombre, gr, gn in zip(["W1", "b1", "W2", "b2"], g_retro, g_num):
    print(f"{nombre}: diferencia máxima = {np.abs(gr - gn).max():.2e}")

Diferencias del orden de $10^{-11}$ o menos: ¡las dos formas dan lo mismo! (No dan exactamente cero porque el gradiente numérico es una aproximación.)

## 6. Entrenar con retropropagación

El descenso de gradiente es el mismo de siempre; solo cambiamos de dónde salen las pendientes:

In [ ]:
import time

def entrenar(X, Y, n_ocultas, alfa, pasos, semilla=0):
    params = iniciar(X.shape[1], n_ocultas, Y.shape[1], semilla)
    historia = []
    for paso in range(pasos):
        grads = retropropagacion(X, Y, params)
        for A, G in zip(params, grads):
            A -= alfa * G
        historia.append(error(X, Y, params))
    return params, historia

inicio = time.time()
params, historia = entrenar(X, Y, n_ocultas=3, alfa=2.0, pasos=3000)
print(f"Tiempo: {time.time() - inicio:.2f} s")
print("Salida de la red:", adelante(X, params)[1].ravel().round(3))
print("Error final:", round(historia[-1], 5))

✏️ *En ML 9, el mismo entrenamiento tardaba más de un segundo. ¿Cuántas veces más rápido es ahora?* La diferencia crece muchísimo con redes más grandes, como veremos.

## 7. Una red que lee dígitos

Usemos los 1797 dígitos escritos a mano de 8 × 8 píxeles (ML 4 y ML 6). Ahora la red tendrá:

- **64 entradas** (un píxel cada una, divididas por 16 para que queden entre 0 y 1);
- **32 neuronas ocultas**;
- **10 salidas**, una por dígito. La respuesta correcta se escribe como una fila con un 1 en la posición del dígito y 0 en las demás (se llama codificación *one-hot*). Por ejemplo, el 3 es `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`.

La predicción es la salida **más alta** (`argmax`). Y, como aprendimos en ML 7, separamos datos de entrenamiento y de prueba.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digitos = load_digits()
Xd = digitos.data / 16
X_ent, X_pru, y_ent, y_pru = train_test_split(Xd, digitos.target, test_size=0.3, random_state=0)
Y_ent = np.eye(10)[y_ent]                  # one-hot: la fila y_ent de la matriz identidad

print("Dígito:", y_ent[0], "→ one-hot:", Y_ent[0])
print("Entrenamiento:", X_ent.shape, "  Prueba:", X_pru.shape)
print("Parámetros de la red:", 64 * 32 + 32 + 32 * 10 + 10)

Primero, comparemos cuánto tarda **un solo paso** con cada método:

In [ ]:
p = iniciar(64, 32, 10, semilla=0)

inicio = time.time(); retropropagacion(X_ent, Y_ent, p); t_retro = time.time() - inicio
inicio = time.time(); gradiente_numerico(X_ent, Y_ent, p); t_num = time.time() - inicio

print(f"Retropropagación: {t_retro * 1000:8.1f} ms por paso")
print(f"Gradiente numérico: {t_num * 1000:6.0f} ms por paso")
print(f"La retropropagación fue unas {t_num / t_retro:.0f} veces más rápida")

Para 2000 pasos de entrenamiento, el gradiente numérico tardaría casi una hora. Con retropropagación, unos segundos:

In [ ]:
inicio = time.time()
params_d, historia_d = entrenar(X_ent, Y_ent, n_ocultas=32, alfa=5.0, pasos=2000)
print(f"Tiempo de entrenamiento: {time.time() - inicio:.1f} s")

def predecir(X, params):
    return adelante(X, params)[1].argmax(axis=1)

print(f"Exactitud en entrenamiento: {np.mean(predecir(X_ent, params_d) == y_ent):.1%}")
print(f"Exactitud en prueba:        {np.mean(predecir(X_pru, params_d) == y_pru):.1%}")

plt.plot(historia_d)
plt.xlabel("paso"); plt.ylabel("error")
plt.title("Curva de aprendizaje con los dígitos")
plt.grid(alpha=0.3)
plt.show()

¡Una red escrita por nosotros, en unas 20 líneas de numpy, reconoce dígitos que nunca vio con más del 95 % de exactitud! Miremos algunos de sus errores:

In [ ]:
pred = predecir(X_pru, params_d)
errores = np.where(pred != y_pru)[0]
print("Errores en prueba:", len(errores), "de", len(y_pru))

fig, ejes = plt.subplots(1, min(8, len(errores)), figsize=(14, 2.5))
for eje, i in zip(np.atleast_1d(ejes), errores[:8]):
    eje.imshow(X_pru[i].reshape(8, 8), cmap="gray_r")
    eje.set_title(f"real {y_pru[i]}, dijo {pred[i]}")
    eje.axis("off")
plt.show()

✏️ *¿Te habrías equivocado tú con alguno de estos? Recuerda que son imágenes de solo 8 × 8 píxeles.*

## 8. ⚠️ El desvanecimiento del gradiente

Cada vez que la culpa atraviesa una sigmoide **hacia atrás**, se multiplica por $\sigma'(z) \le 0.25$. En una red con muchas capas, la culpa se multiplica por números pequeños una y otra vez:

In [ ]:
for capas in [1, 2, 5, 10, 20]:
    print(f"Atravesando {capas:2} capa(s): la culpa se multiplica como mucho por 0.25^{capas} = {0.25 ** capas:.2e}")

Con 10 capas, las primeras reciben una culpa un millón de veces más pequeña: casi no aprenden. Este problema, el **desvanecimiento del gradiente**, frenó durante años las redes profundas.

Una solución muy usada es cambiar la sigmoide de las capas ocultas por la función **ReLU**: $\text{ReLU}(z) = \max(0, z)$. Su pendiente es **1** cuando $z > 0$, así que no encoge la culpa. La usaremos en ML 11 con TensorFlow, que además calcula la retropropagación **automáticamente** para cualquier red que diseñemos.

## 9. Errores comunes

**Error 1: olvidar la transpuesta.** En la vuelta, las formas deben encajar. Si escribes `H @ dZ2` en lugar de `H.T @ dZ2`:

In [ ]:
H, S = adelante(X, params)
dZ2 = (S - Y) * S * (1 - S)
print("H:", H.shape, "  dZ2:", dZ2.shape)
H @ dZ2

Truco: `dW` siempre debe tener la **misma forma** que `W`. Si `W2` es (3, 1), `dW2` también; y la única forma de obtener (3, 1) a partir de `H` (4, 3) y `dZ2` (4, 1) es `H.T @ dZ2`.

**Error 2: olvidar la pendiente de la sigmoide** (`* S * (1 - S)`). El código no da error, ¡pero aprende mal! Por eso se hace la comprobación numérica.

**Error 3: el signo.** El descenso de gradiente **resta**: `A -= alfa * G`. Con `+=` el error sube.

**Error 4: no normalizar las entradas.** Con los píxeles entre 0 y 16 (sin dividir), las sigmoides se saturan y la pendiente se vuelve casi 0.

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …calculas `d_sigmoide(10)` y `d_sigmoide(-10)`? ¿Qué le pasa a la culpa que atraviesa una neurona así?
2. …cambias `A -= alfa * G` por `A += alfa * G` en `entrenar` y entrenas el XOR?
3. …quitas `* S * (1 - S)` en `retropropagacion` y vuelves a hacer la comprobación de la sección 5?
4. …entrenas los dígitos **sin** dividir por 16?
5. …usas `h = 0.1` en `gradiente_numerico`? ¿Siguen coincidiendo las dos formas?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). Regla de la cadena a mano.** Calcula la pendiente de $y = (2x + 3)^3$ en $x = 1$ usando la regla de la cadena. Compruébala con la pendiente numérica.

**Ejercicio 2 (fácil). Una neurona a mano.** Con $x = 1$, $y = 0$, $w = 2$ y $b = -1$, calcula con calculadora (paso a paso, como en la sección 3) $z$, $a$, $E$, $dE/dw$ y $dE/db$. Luego da **un** paso de descenso de gradiente con $\alpha = 1$. ¿Bajó el error?

**Ejercicio 3 (medio). Las medias lunas, ahora rápido.** Entrena la red de las medias lunas de ML 9 (`make_moons(200, noise=0.15, random_state=0)`, 8 neuronas ocultas) con `entrenar`, pero ahora con 20 000 pasos y `alfa=1.0`. (Recuerda escribir `y` como columna: `Y = yl.reshape(-1, 1)`.) ¿Cuánto tarda? ¿Qué exactitud alcanza comparada con el 92 % de ML 9?

**Ejercicio 4 (medio). ¿Cuántas neuronas ocultas?** Entrena la red de dígitos con 4, 8, 16, 32 y 64 neuronas ocultas (2000 pasos, `alfa=5.0`) y grafica la exactitud en entrenamiento y en prueba. ¿Ves sobreajuste en algún punto (ML 7)?

**Ejercicio 5 (avanzado). Dos capas ocultas.** Extiende `iniciar`, `adelante` y `retropropagacion` para una red con **dos** capas ocultas (64 → 32 → 16 → 10). Sigue el mismo patrón: cada capa hacia atrás recibe la culpa de la siguiente a través de sus pesos. Comprueba tus fórmulas con `gradiente_numerico` en el XOR antes de entrenar con los dígitos.

## 🏆 Reto opcional: ReLU

Cambia la activación de la **capa oculta** por ReLU, $\max(0, z)$, cuya pendiente es 1 si $z > 0$ y 0 si no (la salida sigue con sigmoide).

1. ¿Qué cambia en `adelante`? ¿Y en `retropropagacion`? (Pista: necesitas guardar $Z_1 = X W_1 + b_1$ para saber dónde era positivo; `np.maximum(0, Z1)` y `(Z1 > 0)` te ayudan.)
2. Comprueba tus fórmulas con el gradiente numérico.
3. Entrena los dígitos con ReLU. Con ReLU suele hacer falta una tasa de aprendizaje **menor** y pesos iniciales más pequeños: prueba `alfa=0.5` y multiplica los pesos iniciales de `W1` por 0.1. ¿Aprende más rápido que con sigmoide?

## 🎬 Cierre

Hoy aprendiste que:
- La **regla de la cadena** multiplica las tasas de cambio de una cadena de operaciones, como los engranajes.
- La **retropropagación** recorre la red hacia atrás calculando la «culpa» de cada capa y reutilizándola: obtiene **todas** las pendientes con el costo de un par de propagaciones.
- Siempre se comprueba con el **gradiente numérico** en una red pequeña.
- Con retropropagación, una red de 2410 parámetros aprende a leer dígitos en segundos.
- La pendiente de la sigmoide es pequeña, y en redes profundas eso causa el **desvanecimiento del gradiente**; la ReLU ayuda.

**Para pensar:** escribir la retropropagación a mano para cada red nueva es trabajoso y fácil de equivocar. Las librerías modernas lo hacen **solas**: tú describes las capas y ellas calculan todas las pendientes. En la próxima lección usaremos **Keras y TensorFlow** para entrenar, en pocas líneas, una red con el famoso conjunto **MNIST**: 70 000 dígitos de 28 × 28 píxeles. 🚀